# ST554_Project 2
Author: Huiping Zhou
Data: 3/14/2026

# Part I-Creating a Class
In this section, we are going to create a class called `SparkDataCheck` and save it on a .py file. We will test the SparkDataCheck class on the [air](https://www4.stat.ncsu.edu/online/datasets/air.csv) dataset which read in as Spark SQL style data frames.

In [2]:
#import needed modules
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd
from pandas.api.types import is_numeric_dtype
from functools import reduce

In [3]:
# Create a SparkSession named 'my_app'
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('my_app').config("spark.sql.ansi.enabled", "false").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/15 21:47:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


We read in dataset `air` using the method that created in class file. we import the my_class into this notebook first.

In [4]:
import my_class

In [5]:
#you can run this code and this will re-import that class that you have there without having to restart your kernel.
import importlib
importlib.reload(my_class)

<module 'my_class' from '/home/jupyter-hzhou13@ncsu.edu/Project2/my_class.py'>

### Reading in dataset using `from_spark` method
We use the `from_spark` method to read in the dataset and dispaly the first 5 rows using the `'show()` method.

In [5]:
sql_df = my_class.SparkDataCheck.from_spark(
    spark,
    "air.csv",
    format="csv",
    header=True,
    inferSchema=True,
    sep=",")
sql_df.show(5)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-15 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-15 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-15 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-15 21:00:00|   2.2|       137

26/03/15 21:47:18 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


Based on the output above, the `from_spark` method successfully read the data correctly.

In [7]:
#check the data types of clomuns
col_types = sql_df.dtypes
col_types

[('_c0', 'int'),
 ('Date', 'string'),
 ('Time', 'timestamp'),
 ('CO(GT)', 'double'),
 ('PT08.S1(CO)', 'int'),
 ('NMHC(GT)', 'int'),
 ('C6H6(GT)', 'double'),
 ('PT08.S2(NMHC)', 'int'),
 ('NOx(GT)', 'int'),
 ('PT08.S3(NOx)', 'int'),
 ('NO2(GT)', 'int'),
 ('PT08.S4(NO2)', 'int'),
 ('PT08.S5(O3)', 'int'),
 ('T', 'double'),
 ('RH', 'double'),
 ('AH', 'double')]

We will rename the columns containing dots (.) to avoid issues in Spark DataFrames.

In [6]:
sql_air = (sql_df
        .withColumnRenamed('PT08.S1(CO)', 'sensor_CO')
        .withColumnRenamed('PT08.S2(NMHC)', 'sensor_NMHC')
        .withColumnRenamed('PT08.S3(NOx)', 'sensor_NOx')
        .withColumnRenamed('PT08.S4(NO2)', 'sensor_NO2')
        .withColumnRenamed('PT08.S5(O3)', 'sensor_O3'))
sql_air.show(5)

+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+
|_c0|     Date|               Time|CO(GT)|sensor_CO|NMHC(GT)|C6H6(GT)|sensor_NMHC|NOx(GT)|sensor_NOx|NO2(GT)|sensor_NO2|sensor_O3|   T|  RH|    AH|
+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+
|  0|3/10/2004|2026-03-15 18:00:00|   2.6|     1360|     150|    11.9|       1046|    166|      1056|    113|      1692|     1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-15 19:00:00|   2.0|     1292|     112|     9.4|        955|    103|      1174|     92|      1559|      972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-15 20:00:00|   2.2|     1402|      88|     9.0|        939|    131|      1140|    114|      1555|     1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-15 21:00:00|   2.2|     1376|      80|     9.2|        948|    172|      1092|    122|   

26/03/15 21:47:23 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


Since we know that this dataset uses -200 to code missing values, we will check whether -200 appears in any column by generating basic summary statistics using the `summary()` method.

In [7]:
num_cols = ['CO(GT)', 'sensor_CO', 'NMHC(GT)', 'C6H6(GT)', 'sensor_NMHC', 'NOx(GT)', 'sensor_NOx', 'NO2(GT)',  'sensor_NO2', 'sensor_O3', 'T', 'RH', 'AH']
sql_air.select(*num_cols).summary('count', 'mean', 'stddev', 'min', 'max').show()

26/03/15 21:47:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+
|summary|            CO(GT)|         sensor_CO|           NMHC(GT)|          C6H6(GT)|      sensor_NMHC|          NOx(GT)|        sensor_NOx|           NO2(GT)|        sensor_NO2|         sensor_O3|                T|               RH|                AH|
+-------+------------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+
|  count|              9357|              9357|               9357|              9357|             9357|             9357|              9357|              9357|              9357|              9357|             9357|             9357|    

The basic summary statistics show that all numeric variables have a minimum value of -200, which is the sentinel value used in this dataset to represent missing observations. Then we will replace the coded missing values(-200) with NULL.

In [8]:
#replace the coded missing value (-200) with NULL in all numeric columns
sql_air =sql_air.na.replace({-200: None})
sql_air.show()

+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+
|_c0|     Date|               Time|CO(GT)|sensor_CO|NMHC(GT)|C6H6(GT)|sensor_NMHC|NOx(GT)|sensor_NOx|NO2(GT)|sensor_NO2|sensor_O3|   T|  RH|    AH|
+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+
|  0|3/10/2004|2026-03-15 18:00:00|   2.6|     1360|     150|    11.9|       1046|    166|      1056|    113|      1692|     1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-15 19:00:00|   2.0|     1292|     112|     9.4|        955|    103|      1174|     92|      1559|      972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-15 20:00:00|   2.2|     1402|      88|     9.0|        939|    131|      1140|    114|      1555|     1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-15 21:00:00|   2.2|     1376|      80|     9.2|        948|    172|      1092|    122|   

26/03/15 21:47:40 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


Next, we will show examples of calling each method from the my_class file on this object, beginning with 
### Testing the `check_numeric_range method`

In [23]:
# Create a SparkDataCheck instance from the existing sql_air DataFrame
sql_air_checker = my_class.SparkDataCheck(sql_air)

# provide two bounds (2,5)
sql_air_checker.check_numeric_range(column = 'CO(GT)', lower = 2, upper = 5, new_column = 'new_CO(GT)').show()

+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+----------+
|_c0|     Date|               Time|CO(GT)|sensor_CO|NMHC(GT)|C6H6(GT)|sensor_NMHC|NOx(GT)|sensor_NOx|NO2(GT)|sensor_NO2|sensor_O3|   T|  RH|    AH|new_CO(GT)|
+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+----------+
|  0|3/10/2004|2026-03-15 18:00:00|   2.6|     1360|     150|    11.9|       1046|    166|      1056|    113|      1692|     1268|13.6|48.9|0.7578|      true|
|  1|3/10/2004|2026-03-15 19:00:00|   2.0|     1292|     112|     9.4|        955|    103|      1174|     92|      1559|      972|13.3|47.7|0.7255|      true|
|  2|3/10/2004|2026-03-15 20:00:00|   2.2|     1402|      88|     9.0|        939|    131|      1140|    114|      1555|     1074|11.9|54.0|0.7502|      true|
|  3|3/10/2004|2026-03-15 21:00:00|   2.2|    

26/03/15 22:32:54 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


When the lower and upper bounds are provided, we expect the function to return a DataFrame with a new Boolean column.We can also see that when the data is missing, the appended Boolean column returns NULL.The output above shows that the results meet this expectation.

In [22]:
#provide one bound
sql_air_checker.check_numeric_range(column = 'CO(GT)', lower =2 , upper =None, new_column = 'new_CO(GT)').show()

+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+----------+
|_c0|     Date|               Time|CO(GT)|sensor_CO|NMHC(GT)|C6H6(GT)|sensor_NMHC|NOx(GT)|sensor_NOx|NO2(GT)|sensor_NO2|sensor_O3|   T|  RH|    AH|new_CO(GT)|
+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+----------+
|  0|3/10/2004|2026-03-15 18:00:00|   2.6|     1360|     150|    11.9|       1046|    166|      1056|    113|      1692|     1268|13.6|48.9|0.7578|      true|
|  1|3/10/2004|2026-03-15 19:00:00|   2.0|     1292|     112|     9.4|        955|    103|      1174|     92|      1559|      972|13.3|47.7|0.7255|      true|
|  2|3/10/2004|2026-03-15 20:00:00|   2.2|     1402|      88|     9.0|        939|    131|      1140|    114|      1555|     1074|11.9|54.0|0.7502|      true|
|  3|3/10/2004|2026-03-15 21:00:00|   2.2|    

26/03/15 22:30:53 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


When only one bound is provided (in this case, the lower bound), the check is applied only to the lower side. We can also see that when the data is missing, the appended Boolean column returns NULL.The output above confirms that this works as intended.

In [21]:
# no bound provided
sql_air_checker.check_numeric_range(column = 'CO(GT)', lower = None, upper = None, new_column = 'new_CO(GT)').show(8)

Error: At least one of 'lower' or 'upper' must be provided.
+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+
|_c0|     Date|               Time|CO(GT)|sensor_CO|NMHC(GT)|C6H6(GT)|sensor_NMHC|NOx(GT)|sensor_NOx|NO2(GT)|sensor_NO2|sensor_O3|   T|  RH|    AH|
+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+
|  0|3/10/2004|2026-03-15 18:00:00|   2.6|     1360|     150|    11.9|       1046|    166|      1056|    113|      1692|     1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-15 19:00:00|   2.0|     1292|     112|     9.4|        955|    103|      1174|     92|      1559|      972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-15 20:00:00|   2.2|     1402|      88|     9.0|        939|    131|      1140|    114|      1555|     1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-15 21:00:00|   2.2|     1376|

26/03/15 22:30:47 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


If no bounds are provided, a message is displayed reminding us that at least one bound is required, and the DataFrame is returned without modification. The output above shows that this works as intended.

In [18]:
#provide a string colume
sql_air_checker.check_numeric_range(column = 'Date').show(8)

+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+
|_c0|     Date|               Time|CO(GT)|sensor_CO|NMHC(GT)|C6H6(GT)|sensor_NMHC|NOx(GT)|sensor_NOx|NO2(GT)|sensor_NO2|sensor_O3|   T|  RH|    AH|
+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+
|  0|3/10/2004|2026-03-15 18:00:00|   2.6|     1360|     150|    11.9|       1046|    166|      1056|    113|      1692|     1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-15 19:00:00|   2.0|     1292|     112|     9.4|        955|    103|      1174|     92|      1559|      972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-15 20:00:00|   2.2|     1402|      88|     9.0|        939|    131|      1140|    114|      1555|     1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-15 21:00:00|   2.2|     1376|      80|     9.2|        948|    172|      1092|    122|   

26/03/15 22:00:27 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


If we accidentally provide a string column, a warning message is displayed indicating that the column is not numeric, and the function returns the DataFrame without modification. The output confirms that this requirement is met.

Across the four test scenarios (1) providing both bounds, (2) providing a single bound, (3) providing no bounds, and (4) providing a string column, the outputs behaved exactly as expected. These results confirm that the check_numeric_range method is implemented correctly.

### Testing `check_string_levels` method

First, we will create a season-level categorical variable using the Date column to further test the `check_string_leaves` method.

In [32]:
# 1) Parse the string to a DateType column
df1 = sql_air.withColumn("parsed_date", F.to_date(F.col("Date"), "M/d/yyyy"))

# 2) Derive month (1-12)
df2 = df1.withColumn("month_num", F.month("parsed_date"))

# 3) Map month to season
df3 = df2.withColumn(
    "season",
    F.when(F.col("month_num").isin(12, 1, 2), F.lit("Winter"))
     .when(F.col("month_num").isin(3, 4, 5), F.lit("Spring"))
     .when(F.col("month_num").isin(6, 7, 8), F.lit("Summer"))
     .when(F.col("month_num").isin(9, 10, 11), F.lit("Fall"))
     .otherwise(F.lit(None))  # stays NULL if date couldn't be parsed
)
# drop the month_num and parsed_date column
df3 = df3.drop("month_num", 'parsed_date')
df3.show(5)

+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+------+
|_c0|     Date|               Time|CO(GT)|sensor_CO|NMHC(GT)|C6H6(GT)|sensor_NMHC|NOx(GT)|sensor_NOx|NO2(GT)|sensor_NO2|sensor_O3|   T|  RH|    AH|season|
+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+------+
|  0|3/10/2004|2026-03-15 18:00:00|   2.6|     1360|     150|    11.9|       1046|    166|      1056|    113|      1692|     1268|13.6|48.9|0.7578|Spring|
|  1|3/10/2004|2026-03-15 19:00:00|   2.0|     1292|     112|     9.4|        955|    103|      1174|     92|      1559|      972|13.3|47.7|0.7255|Spring|
|  2|3/10/2004|2026-03-15 20:00:00|   2.2|     1402|      88|     9.0|        939|    131|      1140|    114|      1555|     1074|11.9|54.0|0.7502|Spring|
|  3|3/10/2004|2026-03-15 21:00:00|   2.2|     1376|      80|     9.2|

26/03/15 23:13:29 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


In [33]:
#Create a SparkDataCheck instance from the existing sql_air_season DataFrame
df_season = my_class.SparkDataCheck(df3)

# provide two bounds (2,5)
df_season.check_string_levels(column = 'season', levels = ['Fall'], new_column = 'is_Fall').show(8)

+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+------+-------+
|_c0|     Date|               Time|CO(GT)|sensor_CO|NMHC(GT)|C6H6(GT)|sensor_NMHC|NOx(GT)|sensor_NOx|NO2(GT)|sensor_NO2|sensor_O3|   T|  RH|    AH|season|is_Fall|
+---+---------+-------------------+------+---------+--------+--------+-----------+-------+----------+-------+----------+---------+----+----+------+------+-------+
|  0|3/10/2004|2026-03-15 18:00:00|   2.6|     1360|     150|    11.9|       1046|    166|      1056|    113|      1692|     1268|13.6|48.9|0.7578|Spring|  false|
|  1|3/10/2004|2026-03-15 19:00:00|   2.0|     1292|     112|     9.4|        955|    103|      1174|     92|      1559|      972|13.3|47.7|0.7255|Spring|  false|
|  2|3/10/2004|2026-03-15 20:00:00|   2.2|     1402|      88|     9.0|        939|    131|      1140|    114|      1555|     1074|11.9|54.0|0.7502|Spring|  false|
|  3|3/10/2004|2026-03

26/03/15 23:19:20 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-hzhou13@ncsu.edu/Project2/air.csv


In [12]:
import pandas as pd
import numpy as np
import pyspark.pandas as ps
pdf = pd.read_csv("air.csv", delimiter = ",")
pdf.head()

,Unnamed: 0,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888
